In [ ]:
# %% [markdown]
# # Week 5 – Labels & Features Notebook (MRU/USD)
#
# This notebook implements the Week 5 code work:
# - Confirm data and fixed validation splits (V1 + V2).
# - Generate L1 (future-return-based) and L2 (indicator-based) labels.
# - Run basic diagnostics: class balance + run lengths by segment.
# - Inspect feature correlations on V1-train.
# - Produce Plotly figures (price, returns, labels, correlations) for visual inspection.
#
# All numeric outputs come from `data/processed/mru_usd_ohlc_clean.csv`.

import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go

In [ ]:
# Paths
PROJECT_ROOT = Path("../") 
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_DERIVED = PROJECT_ROOT / "data" / "derived"

CLEAN_FILE = DATA_PROCESSED / "mru_usd_ohlc_clean.csv"

DATA_DERIVED.mkdir(parents=True, exist_ok=True)

# Fixed V1 chronological split (from Week 4 decisions)
V1_TRAIN_START = pd.Timestamp("2000-01-03")
V1_TRAIN_END   = pd.Timestamp("2015-12-31")
V1_VAL_START   = pd.Timestamp("2016-01-01")
V1_VAL_END     = pd.Timestamp("2019-12-31")
V1_TEST_START  = pd.Timestamp("2020-01-01")
V1_TEST_END    = pd.Timestamp("2025-11-25")

# Fixed V2 walk-forward folds (expanding train, non-overlapping test windows)
V2_FOLDS = [
    {
        "name": "F1",
        "train_start": pd.Timestamp("2000-01-03"),
        "train_end":   pd.Timestamp("2009-12-31"),
        "test_start":  pd.Timestamp("2010-01-01"),
        "test_end":    pd.Timestamp("2011-12-31"),
    },
    {
        "name": "F2",
        "train_start": pd.Timestamp("2000-01-03"),
        "train_end":   pd.Timestamp("2011-12-31"),
        "test_start":  pd.Timestamp("2012-01-01"),
        "test_end":    pd.Timestamp("2013-12-31"),
    },
    {
        "name": "F3",
        "train_start": pd.Timestamp("2000-01-03"),
        "train_end":   pd.Timestamp("2013-12-31"),
        "test_start":  pd.Timestamp("2014-01-01"),
        "test_end":    pd.Timestamp("2015-12-31"),
    },
    {
        "name": "F4",
        "train_start": pd.Timestamp("2000-01-03"),
        "train_end":   pd.Timestamp("2015-12-31"),
        "test_start":  pd.Timestamp("2016-01-01"),
        "test_end":    pd.Timestamp("2019-12-31"),
    },
]

# Era blocks for diagnostics (same as Week 4)
ERA_SEGMENTS = {
    "Era1_2000_2004": (pd.Timestamp("2000-01-01"), pd.Timestamp("2004-12-31")),
    "Era2_2005_2009": (pd.Timestamp("2005-01-01"), pd.Timestamp("2009-12-31")),
    "Era3_2010_2014": (pd.Timestamp("2010-01-01"), pd.Timestamp("2014-12-31")),
    "Era4_2015_2019": (pd.Timestamp("2015-01-01"), pd.Timestamp("2019-12-31")),
    "Era5_2020_2025": (pd.Timestamp("2020-01-01"), pd.Timestamp("2025-11-25")),
}

# Default L1 and L2 parameters (from Week 5 plan)
L1_K_DEFAULT = 5
L1_LAMBDA_DEFAULT = 0.75

L2_MA_FAST = 10
L2_MA_SLOW = 50
L2_SLOPE_H = 5

# Core feature set (from label_feature_plan & methodology)
CORE_FEATURE_COLS = [
    "f_r_lag1",
    "f_past_ret_5",
    "f_past_ret_20",
    "f_vol_10",
    "f_vol_30",
]


In [ ]:
# %% 
# Helper functions (subset, basic transforms, labels, features) =====

def subset_by_date(df: pd.DataFrame, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    """Return df restricted to [start, end] inclusive."""
    mask = (df["date"] >= start) & (df["date"] <= end)
    return df.loc[mask].copy()


def ensure_log_price_and_return(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure df has:
    - log_price = ln(close)
    - log_return = log_price.diff()
    """
    df = df.copy()
    if "log_price" not in df.columns:
        df["log_price"] = np.log(df["close"])
    if "log_return" not in df.columns:
        df["log_return"] = df["log_price"].diff()
    return df


def print_segment_counts(df: pd.DataFrame) -> None:
    """Sanity check: number of rows in V1 segments and V2 folds."""
    print("\n=== V1 segments row counts ===")
    segments_v1 = {
        "V1_train": (V1_TRAIN_START, V1_TRAIN_END),
        "V1_val":   (V1_VAL_START, V1_VAL_END),
        "V1_test":  (V1_TEST_START, V1_TEST_END),
    }
    for name, (start, end) in segments_v1.items():
        seg = subset_by_date(df, start, end)
        print(f"{name}: {len(seg)} rows ({start.date()} → {end.date()})")

    print("\n=== V2 folds row counts (train/test) ===")
    for fold in V2_FOLDS:
        train = subset_by_date(df, fold["train_start"], fold["train_end"])
        test  = subset_by_date(df, fold["test_start"], fold["test_end"])
        print(
            f"{fold['name']}: "
            f"train={len(train)} ({fold['train_start'].date()} → {fold['train_end'].date()}), "
            f"test={len(test)} ({fold['test_start'].date()} → {fold['test_end'].date()})"
        )


def generate_L1_labels(
    df: pd.DataFrame,
    train_start: pd.Timestamp,
    train_end: pd.Timestamp,
    k: int = L1_K_DEFAULT,
    lambda_: float = L1_LAMBDA_DEFAULT,
    label_col: str = "y_L1",
    future_return_col: str = "R_t_to_tk",
):
    """
    Generate L1 labels for the full series, following the label_feature_plan:

    - sigma_train: std of daily log returns in [train_start, train_end].
    - Future k-day log return: R_{t→t+k} = log_price.shift(-k) - log_price.
    - Threshold: theta = lambda * sigma_train * sqrt(k).
    - Labels: +1 (bull), 0 (neutral), -1 (bear).
    """
    df_l1 = ensure_log_price_and_return(df)

    train_mask = (df_l1["date"] >= train_start) & (df_l1["date"] <= train_end)
    sigma_train = df_l1.loc[train_mask, "log_return"].dropna().std()

    df_l1[future_return_col] = df_l1["log_price"].shift(-k) - df_l1["log_price"]
    theta = lambda_ * sigma_train * np.sqrt(k)

    df_l1[label_col] = np.nan
    valid_mask = df_l1[future_return_col].notna()

    df_l1.loc[valid_mask & (df_l1[future_return_col] > theta), label_col] = 1
    df_l1.loc[valid_mask & (df_l1[future_return_col] < -theta), label_col] = -1
    df_l1.loc[valid_mask & df_l1[label_col].isna(), label_col] = 0

    print("\n=== L1 parameters ===")
    print(f"k = {k}, lambda = {lambda_}")
    print(f"sigma_train (V1-train) = {sigma_train:.6f}")
    print(f"theta = lambda * sigma_train * sqrt(k) = {theta:.6f}")

    return df_l1, sigma_train, theta


def generate_L2_labels(
    df: pd.DataFrame,
    ma_fast_window: int = L2_MA_FAST,
    ma_slow_window: int = L2_MA_SLOW,
    slope_horizon: int = L2_SLOPE_H,
    label_col: str = "y_L2",
):
    """
    Generate L2 labels (trend heuristic) using:

    - MA_fast: 10-day SMA of log price (past & current only)
    - MA_slow: 50-day SMA of log price
    - MA_slow_slope: MA_slow_t - MA_slow_{t-h}

    Labels:
    - +1 if MA_fast > MA_slow and MA_slow_slope > 0
    - -1 if MA_fast < MA_slow and MA_slow_slope < 0
    -  0 otherwise
    Rows with insufficient history get NaN labels.
    """
    df_l2 = ensure_log_price_and_return(df)

    df_l2["MA_fast"] = (
        df_l2["log_price"]
        .rolling(window=ma_fast_window, min_periods=ma_fast_window)
        .mean()
    )
    df_l2["MA_slow"] = (
        df_l2["log_price"]
        .rolling(window=ma_slow_window, min_periods=ma_slow_window)
        .mean()
    )
    df_l2["MA_slow_slope"] = df_l2["MA_slow"] - df_l2["MA_slow"].shift(slope_horizon)

    df_l2[label_col] = 0
    cond_up = (df_l2["MA_fast"] > df_l2["MA_slow"]) & (df_l2["MA_slow_slope"] > 0)
    cond_down = (df_l2["MA_fast"] < df_l2["MA_slow"]) & (df_l2["MA_slow_slope"] < 0)

    df_l2.loc[cond_up, label_col] = 1
    df_l2.loc[cond_down, label_col] = -1

    # Insufficient history → NaN
    insufficient = df_l2["MA_slow"].isna() | df_l2["MA_slow_slope"].isna()
    df_l2.loc[insufficient, label_col] = np.nan

    print("\n=== L2 parameters ===")
    print(f"MA_fast_window = {ma_fast_window}")
    print(f"MA_slow_window = {ma_slow_window}")
    print(f"slope_horizon = {slope_horizon}")

    return df_l2


def summarize_label_distribution(
    df: pd.DataFrame,
    label_col: str,
    segments_dict: dict,
) -> pd.DataFrame:
    """
    Print label counts + proportions per segment for given label column.
    segments_dict: {segment_name: (start_date, end_date)}
    """
    records = []
    for seg_name, (start, end) in segments_dict.items():
        seg = subset_by_date(df, start, end)
        counts = seg[label_col].value_counts(dropna=True).sort_index()
        total = counts.sum()
        for cls, cnt in counts.items():
            records.append(
                {
                    "segment": seg_name,
                    "label": int(cls),
                    "count": int(cnt),
                    "proportion": float(cnt) / float(total) if total > 0 else np.nan,
                }
            )

    summary_df = pd.DataFrame(records)
    print(f"\n=== Label distribution for {label_col} by segment ===")
    if summary_df.empty:
        print("No labels found for any segment.")
    else:
        print(summary_df.to_string(index=False))
    return summary_df


def compute_run_lengths(
    df: pd.DataFrame,
    label_col: str,
    segment_name: str,
    start: pd.Timestamp,
    end: pd.Timestamp,
) -> pd.DataFrame:
    """
    Compute run-length statistics for each label value in a date segment.
    Returns a DataFrame of individual runs and prints per-class summary stats.
    """
    seg = subset_by_date(df, start, end)
    labels = seg[label_col].dropna().values

    run_records = []
    if labels.size == 0:
        print(f"\n=== Run-length stats for {label_col} in {segment_name} ===")
        print("No labels (all NaN or segment empty).")
        return pd.DataFrame(columns=["label", "run_length"])

    current_label = labels[0]
    run_len = 1
    for val in labels[1:]:
        if val == current_label:
            run_len += 1
        else:
            run_records.append({"label": int(current_label), "run_length": run_len})
            current_label = val
            run_len = 1
    run_records.append({"label": int(current_label), "run_length": run_len})

    runs_df = pd.DataFrame(run_records)

    print(f"\n=== Run-length stats for {label_col} in {segment_name} ===")
    if runs_df.empty:
        print("No runs.")
    else:
        print(runs_df.groupby("label")["run_length"].describe())

    return runs_df


def compute_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute core and  features with anti-leakage .

    Core features:
      - f_r_lag1       : lagged daily log return r_{t-1}
      - f_past_ret_5   : 5-day past cumulative log return (t-5 → t-1)
      - f_past_ret_20  : 20-day past cumulative log return (t-20 → t-1)
      - f_vol_10       : 10-day rolling std of returns up to t-1
      - f_vol_30       : 30-day rolling std of returns up to t-1

      - f_ma_50        : 50-day mean of log price up to t-1
      - f_dist_50      : distance of log price from its 50-day mean
    """
    df_feat = ensure_log_price_and_return(df)

    # One-day lagged log return r_{t-1}
    df_feat["f_r_lag1"] = df_feat["log_return"].shift(1)

    # Past cumulative log returns using log_price up to t-1
    log_p_shift1 = df_feat["log_price"].shift(1)
    df_feat["f_past_ret_5"] = log_p_shift1 - df_feat["log_price"].shift(5)
    df_feat["f_past_ret_20"] = log_p_shift1 - df_feat["log_price"].shift(20)

    # Volatility features: rolling std of returns up to t-1
    ret_shift1 = df_feat["log_return"].shift(1)
    df_feat["f_vol_10"] = ret_shift1.rolling(window=10, min_periods=10).std()
    df_feat["f_vol_30"] = ret_shift1.rolling(window=30, min_periods=30).std()

    log_p_shift1 = df_feat["log_price"].shift(1)
    df_feat["f_ma_50"] = log_p_shift1.rolling(window=50, min_periods=50).mean()
    df_feat["f_dist_50"] = log_p_shift1 - df_feat["f_ma_50"]

    return df_feat


def add_high_vol_indicator(
    df_feat: pd.DataFrame,
    train_start: pd.Timestamp,
    train_end: pd.Timestamp,
    vol_col: str = "f_vol_10",
    quantile: float = 0.7,
    indicator_col: str = "f_high_vol",
):
    """
    Add a binary high-volatility indicator based on a quantile of vol_col in V1-train.
    """
    df_feat = df_feat.copy()
    train_mask = (df_feat["date"] >= train_start) & (df_feat["date"] <= train_end)
    q = df_feat.loc[train_mask, vol_col].dropna().quantile(quantile)

    df_feat[indicator_col] = 0
    df_feat.loc[df_feat[vol_col] > q, indicator_col] = 1

    print("\n=== High-vol indicator threshold ===")
    print(f"{vol_col} {quantile*100:.1f}th percentile on V1-train = {q:.6f}")
    return df_feat, q


def correlation_diagnostics(
    df_feat: pd.DataFrame,
    feature_cols: list,
    start: pd.Timestamp,
    end: pd.Timestamp,
    threshold: float = 0.95,
):
    """
    Compute correlation matrix for feature_cols on [start, end] and highlight
    pairs with |correlation| >= threshold.
    """
    mask = (df_feat["date"] >= start) & (df_feat["date"] <= end)
    train = df_feat.loc[mask, feature_cols].dropna()

    print("\n=== Core feature correlation matrix (V1-train) ===")
    if train.empty:
        print("No rows with complete features in this segment.")
        return pd.DataFrame(), []

    corr = train.corr()
    print(corr.to_string())

    high_pairs = []
    for i, col_i in enumerate(feature_cols):
        for j, col_j in enumerate(feature_cols):
            if j <= i:
                continue
            val = corr.loc[col_i, col_j]
            if abs(val) >= threshold:
                high_pairs.append((col_i, col_j, float(val)))

    if high_pairs:
        print(f"\nPairs with |correlation| >= {threshold}:")
        for col_i, col_j, val in high_pairs:
            print(f"{col_i} vs {col_j}: {val:.3f}")
    else:
        print(f"\nNo feature pairs with |correlation| >= {threshold}.")

    return corr, high_pairs


In [ ]:
# %%
# =====  Load cleaned data, basic checks, Plotly overview =====

print("=== Loading cleaned MRU/USD data for Week 5 ===")
df = pd.read_csv(CLEAN_FILE)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print("\nHead:")
print(df.head())
print("\nTail:")
print(df.tail())

print("\nDate range and length:")
print(f"{df['date'].min().date()} → {df['date'].max().date()}, {len(df)} rows")

print_segment_counts(df)

# Enrich with log_price and log_return for quick inspection
df = ensure_log_price_and_return(df)

print("\nLog return descriptive statistics (whole series):")
print(df["log_return"].describe())

# --- Plotly: price level over time ---
fig_price = px.line(
    df,
    x="date",
    y="close",
    title="MRU/USD – Close Price (Cleaned Series)",
    labels={"date": "Date", "close": "Close (MRU/USD)"},
)
fig_price.update_layout(height=400)
fig_price.show()

# --- Plotly: histogram of daily log returns ---
fig_ret_hist = px.histogram(
    df.dropna(subset=["log_return"]),
    x="log_return",
    nbins=100,
    title="Histogram of Daily Log Returns (Cleaned Series)",
)
fig_ret_hist.update_layout(height=400)
fig_ret_hist.show()

=== Loading cleaned MRU/USD data for Week 5 ===

Head:
        date    open    high     low   close
0 2000-01-03  22.525  22.525  22.525  22.525
1 2000-01-04  22.525  22.525  22.525  22.525
2 2000-01-05  22.525  22.525  22.525  22.525
3 2000-01-06  22.525  22.525  22.525  22.525
4 2000-01-07  22.530  22.530  22.530  22.530

Tail:
           date   open    high     low  close
6747 2025-11-19  39.84  40.050  39.640  39.70
6748 2025-11-20  39.70  40.010  39.581  39.86
6749 2025-11-21  39.86  39.860  39.581  39.86
6750 2025-11-24  39.86  40.010  39.592  39.75
6751 2025-11-25  39.76  39.994  39.663  39.94

Date range and length:
2000-01-03 → 2025-11-25, 6752 rows

=== V1 segments row counts ===
V1_train: 4169 rows (2000-01-03 → 2015-12-31)
V1_val: 1043 rows (2016-01-01 → 2019-12-31)
V1_test: 1540 rows (2020-01-01 → 2025-11-25)

=== V2 folds row counts (train/test) ===
F1: train=2604 (2000-01-03 → 2009-12-31), test=521 (2010-01-01 → 2011-12-31)
F2: train=3125 (2000-01-03 → 2011-12-31), test=

In [ ]:
# %%
# =====  L1 label generation, saving & diagnostics + Plotly views =====

df_l1, sigma_train_L1, theta_L1 = generate_L1_labels(
    df,
    train_start=V1_TRAIN_START,
    train_end=V1_TRAIN_END,
    k=L1_K_DEFAULT,
    lambda_=L1_LAMBDA_DEFAULT,
    label_col="y_L1",
    future_return_col="R_t_to_tk",
)

# Save compact L1 label file for later modules
l1_out_path = DATA_DERIVED / "mru_usd_labels_L1_k5_l075.csv"
cols_l1_save = ["date", "close", "log_return", "R_t_to_tk", "y_L1"]
df_l1[cols_l1_save].to_csv(l1_out_path, index=False)
print(f"\nSaved L1 labels to: {l1_out_path}")

# Label distribution by V1 segments + eras
segments_for_labels = {
    "V1_train": (V1_TRAIN_START, V1_TRAIN_END),
    "V1_val":   (V1_VAL_START, V1_VAL_END),
    "V1_test":  (V1_TEST_START, V1_TEST_END),
}
segments_for_labels.update(ERA_SEGMENTS)

l1_label_summary = summarize_label_distribution(df_l1, "y_L1", segments_for_labels)

# Run-length diagnostics for L1 on V1-train
l1_runs_v1_train = compute_run_lengths(
    df_l1,
    label_col="y_L1",
    segment_name="V1_train",
    start=V1_TRAIN_START,
    end=V1_TRAIN_END,
)

# --- Plotly: L1 labels over time (color-coded) ---
df_l1_plot = df_l1.dropna(subset=["y_L1"]).copy()
df_l1_plot["y_L1_str"] = df_l1_plot["y_L1"].map({-1: "Down", 0: "Neutral", 1: "Up"})

fig_l1 = px.scatter(
    df_l1_plot,
    x="date",
    y="close",
    color="y_L1_str",
    title="L1 Regime Labels vs Close Price",
    labels={"date": "Date", "close": "Close", "y_L1_str": "L1 Regime"},
    opacity=0.7,
)
fig_l1.update_traces(marker=dict(size=4))
fig_l1.update_layout(height=450)
fig_l1.show()



=== L1 parameters ===
k = 5, lambda = 0.75
sigma_train (V1-train) = 0.012360
theta = lambda * sigma_train * sqrt(k) = 0.020729

Saved L1 labels to: ..\data\derived\mru_usd_labels_L1_k5_l075.csv

=== Label distribution for y_L1 by segment ===
       segment  label  count  proportion
      V1_train     -1    179    0.042936
      V1_train      0   3812    0.914368
      V1_train      1    178    0.042696
        V1_val     -1      6    0.005753
        V1_val      0   1029    0.986577
        V1_val      1      8    0.007670
       V1_test     -1     14    0.009121
       V1_test      0   1503    0.979153
       V1_test      1     18    0.011726
Era1_2000_2004     -1     54    0.041538
Era1_2000_2004      0   1184    0.910769
Era1_2000_2004      1     62    0.047692
Era2_2005_2009     -1     45    0.034509
Era2_2005_2009      0   1216    0.932515
Era2_2005_2009      1     43    0.032975
Era3_2010_2014     -1     49    0.037577
Era3_2010_2014      0   1222    0.937117
Era3_2010_2014     

In [ ]:
# %%
# ===== label generation, saving & diagnostics + Plotly views =====

df_l2 = generate_L2_labels(
    df,
    ma_fast_window=L2_MA_FAST,
    ma_slow_window=L2_MA_SLOW,
    slope_horizon=L2_SLOPE_H,
    label_col="y_L2",
)

# Save compact L2 label file for later modules
l2_out_path = DATA_DERIVED / "mru_usd_labels_L2_ma10_50_h5.csv"
cols_l2_save = ["date", "close", "MA_fast", "MA_slow", "MA_slow_slope", "y_L2"]
df_l2[cols_l2_save].to_csv(l2_out_path, index=False)
print(f"\nSaved L2 labels to: {l2_out_path}")

# Label distribution for L2
l2_label_summary = summarize_label_distribution(df_l2, "y_L2", segments_for_labels)

# Run-length diagnostics for L2 on V1-train
l2_runs_v1_train = compute_run_lengths(
    df_l2,
    label_col="y_L2",
    segment_name="V1_train",
    start=V1_TRAIN_START,
    end=V1_TRAIN_END,
)

# --- Plotly: L2 labels vs close price ---
df_l2_plot = df_l2.dropna(subset=["y_L2"]).copy()
df_l2_plot["y_L2_str"] = df_l2_plot["y_L2"].map({-1: "Down", 0: "Neutral", 1: "Up"})

fig_l2 = px.scatter(
    df_l2_plot,
    x="date",
    y="close",
    color="y_L2_str",
    title="L2 Regime Labels vs Close Price",
    labels={"date": "Date", "close": "Close", "y_L2_str": "L2 Regime"},
    opacity=0.7,
)
fig_l2.update_traces(marker=dict(size=4))
fig_l2.update_layout(height=450)
fig_l2.show()



=== L2 parameters ===
MA_fast_window = 10
MA_slow_window = 50
slope_horizon = 5

Saved L2 labels to: ..\data\derived\mru_usd_labels_L2_ma10_50_h5.csv

=== Label distribution for y_L2 by segment ===
       segment  label  count  proportion
      V1_train     -1   1409    0.342406
      V1_train      0    767    0.186391
      V1_train      1   1939    0.471203
        V1_val     -1    147    0.140940
        V1_val      0    188    0.180249
        V1_val      1    708    0.678811
       V1_test     -1    457    0.296753
       V1_test      0    268    0.174026
       V1_test      1    815    0.529221
Era1_2000_2004     -1    429    0.344302
Era1_2000_2004      0    216    0.173355
Era1_2000_2004      1    601    0.482343
Era2_2005_2009     -1    504    0.386503
Era2_2005_2009      0    259    0.198620
Era2_2005_2009      1    541    0.414877
Era3_2010_2014     -1    413    0.316718
Era3_2010_2014      0    242    0.185583
Era3_2010_2014      1    649    0.497699
Era4_2015_2019     -1 

In [ ]:
# %%
# ===== Feature computation, high-vol indicator, saving =====

df_feat = compute_features(df)

# Add high-volatility indicator based on V1-train f_vol_10 distribution
df_feat, high_vol_threshold = add_high_vol_indicator(
    df_feat,
    train_start=V1_TRAIN_START,
    train_end=V1_TRAIN_END,
    vol_col="f_vol_10",
    quantile=0.7,
    indicator_col="f_high_vol",
)
features_out_path = DATA_DERIVED / "mru_usd_features_all_week5.csv"
df_feat.to_csv(features_out_path, index=False)
print(f"\nSaved features (core + optional) to: {features_out_path}")

print("\nNon-null counts for core features (whole series):")
print(df_feat[CORE_FEATURE_COLS].notna().sum())

df_vol_plot = df_feat.dropna(subset=["f_vol_10"]).copy()
fig_vol = px.line(
    df_vol_plot,
    x="date",
    y="f_vol_10",
    title="10-day Rolling Volatility of Log Returns (f_vol_10)",
    labels={"date": "Date", "f_vol_10": "10-day rolling σ (log returns)"},
)
fig_vol.update_layout(height=400)
fig_vol.show()


=== High-vol indicator threshold ===
f_vol_10 70.0th percentile on V1-train = 0.005555

Saved features (core + optional) to: ..\data\derived\mru_usd_features_all_week5.csv

Non-null counts for core features (whole series):
f_r_lag1         6750
f_past_ret_5     6747
f_past_ret_20    6732
f_vol_10         6741
f_vol_30         6721
dtype: int64


In [ ]:
# %%
# ===== Feature correlation diagnostics (V1-train) + Plotly heatmap =====

corr_core, high_pairs_core = correlation_diagnostics(
    df_feat,
    feature_cols=CORE_FEATURE_COLS,
    start=V1_TRAIN_START,
    end=V1_TRAIN_END,
    threshold=0.95,
)

# Plotly heatmap of the core feature correlation matrix
if not corr_core.empty:
    fig_corr = px.imshow(
        corr_core,
        text_auto=True,
        aspect="auto",
        title="Core Feature Correlation Matrix (V1-train)",
        labels=dict(color="Correlation"),
    )
    fig_corr.update_layout(height=450)
    fig_corr.show()



=== Core feature correlation matrix (V1-train) ===
               f_r_lag1  f_past_ret_5  f_past_ret_20  f_vol_10  f_vol_30
f_r_lag1       1.000000      0.396678       0.250985  0.047713  0.023451
f_past_ret_5   0.396678      1.000000       0.420048  0.106848  0.050067
f_past_ret_20  0.250985      0.420048       1.000000  0.044922  0.082938
f_vol_10       0.047713      0.106848       0.044922  1.000000  0.754014
f_vol_30       0.023451      0.050067       0.082938  0.754014  1.000000

No feature pairs with |correlation| >= 0.95.


In [ ]:
# %%
# ===== Final Week 5 code  =====

print("\n=== Week 5 notebook – Code Phase Summary ===")
print(f"L1 labels file: {l1_out_path}")
print(f"L2 labels file: {l2_out_path}")
print(f"Features file : {features_out_path}")
print(f"L1 sigma_train (V1-train) = {sigma_train_L1:.6f}")
print(f"L1 theta (k={L1_K_DEFAULT}, lambda={L1_LAMBDA_DEFAULT}) = {theta_L1:.6f}")
print("Core feature columns:", CORE_FEATURE_COLS)
print("High-correlation feature pairs on V1-train (|ρ| ≥ 0.95):")
for col_i, col_j, val in high_pairs_core:
    print(f"  {col_i} vs {col_j}: ρ = {val:.3f}")
if not high_pairs_core:
    print("  None (no pairs above threshold).")




=== Week 5 notebook – Code Phase Summary ===
L1 labels file: ..\data\derived\mru_usd_labels_L1_k5_l075.csv
L2 labels file: ..\data\derived\mru_usd_labels_L2_ma10_50_h5.csv
Features file : ..\data\derived\mru_usd_features_all_week5.csv
L1 sigma_train (V1-train) = 0.012360
L1 theta (k=5, lambda=0.75) = 0.020729
Core feature columns: ['f_r_lag1', 'f_past_ret_5', 'f_past_ret_20', 'f_vol_10', 'f_vol_30']
High-correlation feature pairs on V1-train (|ρ| ≥ 0.95):
  None (no pairs above threshold).

Please copy the printed tables and summaries into your Week 5 docs when we move to the documentation phase.


In [ ]:
# %% 
# =====  L1 alternatives: sigma-based λ grid (same k=5) =====

# We reuse:
# - df (cleaned data with log_price/log_return)
# - generate_L1_labels
# - summarize_label_distribution
# - compute_run_lengths
# - segments_for_labels (defined in Cell 4)

L1_LAMBDA_GRID = [0.5, 0.6, 0.75]  # 0.75 is the current baseline

l1_alt_results = {}  # to store summaries if needed later

for lambda_ in L1_LAMBDA_GRID:
    label_col = f"y_L1_l{int(lambda_ * 100):03d}"
    future_ret_col = f"R_t_to_tk_l{int(lambda_ * 100):03d}"
    print("\n" + "=" * 80)
    print(f"Running L1 with k={L1_K_DEFAULT}, lambda={lambda_} (label_col={label_col})")
    print("=" * 80)

    df_l1_alt, sigma_train_alt, theta_alt = generate_L1_labels(
        df,
        train_start=V1_TRAIN_START,
        train_end=V1_TRAIN_END,
        k=L1_K_DEFAULT,
        lambda_=lambda_,
        label_col=label_col,
        future_return_col=future_ret_col,
    )

    # Label distribution by V1 segments + eras
    l1_alt_label_summary = summarize_label_distribution(
        df_l1_alt,
        label_col=label_col,
        segments_dict=segments_for_labels,
    )

    # Run-length diagnostics on V1-train
    l1_alt_runs_v1_train = compute_run_lengths(
        df_l1_alt,
        label_col=label_col,
        segment_name=f"V1_train (lambda={lambda_})",
        start=V1_TRAIN_START,
        end=V1_TRAIN_END,
    )

    # Store for potential later use
    l1_alt_results[lambda_] = {
        "df_l1": df_l1_alt,
        "sigma_train": sigma_train_alt,
        "theta": theta_alt,
        "label_summary": l1_alt_label_summary,
        "runs_v1_train": l1_alt_runs_v1_train,
    }




Running L1 with k=5, lambda=0.5 (label_col=y_L1_l050)

=== L1 parameters ===
k = 5, lambda = 0.5
sigma_train (V1-train) = 0.012360
theta = lambda * sigma_train * sqrt(k) = 0.013819

=== Label distribution for y_L1_l050 by segment ===
       segment  label  count  proportion
      V1_train     -1    300    0.071960
      V1_train      0   3588    0.860638
      V1_train      1    281    0.067402
        V1_val     -1     15    0.014382
        V1_val      0   1009    0.967402
        V1_val      1     19    0.018217
       V1_test     -1     23    0.014984
       V1_test      0   1474    0.960261
       V1_test      1     38    0.024756
Era1_2000_2004     -1     65    0.050000
Era1_2000_2004      0   1150    0.884615
Era1_2000_2004      1     85    0.065385
Era2_2005_2009     -1     91    0.069785
Era2_2005_2009      0   1134    0.869632
Era2_2005_2009      1     79    0.060583
Era3_2010_2014     -1     94    0.072086
Era3_2010_2014      0   1143    0.876534
Era3_2010_2014      1     6

In [ ]:
# %%
# ===== L1 quantile-based alternative (L1_Q) =====

def generate_L1_quantile_labels(
    df: pd.DataFrame,
    train_start: pd.Timestamp,
    train_end: pd.Timestamp,
    k: int = L1_K_DEFAULT,
    q_low: float = 0.2,
    q_high: float = 0.8,
    label_col: str = "y_L1_Q",
    future_return_col: str = "R_t_to_tk_Q",
):
    """
    Quantile-based L1 alternative:

    - Compute k-day future log return: R_{t→t+k} = log_price.shift(-k) - log_price.
    - On V1-train only, compute quantiles q_low, q_high of R.
    - Labels:
        +1 if R > q_high,
        -1 if R < q_low,
         0 otherwise.
    """
    df_q = ensure_log_price_and_return(df)

    # Future k-day log return
    df_q[future_return_col] = df_q["log_price"].shift(-k) - df_q["log_price"]

    # Estimate quantiles on V1-train only
    train_mask = (df_q["date"] >= train_start) & (df_q["date"] <= train_end)
    train_R = df_q.loc[train_mask, future_return_col].dropna()

    ql = train_R.quantile(q_low)
    qh = train_R.quantile(q_high)

    # Initialize labels
    df_q[label_col] = np.nan
    valid_mask = df_q[future_return_col].notna()

    df_q.loc[valid_mask & (df_q[future_return_col] > qh), label_col] = 1
    df_q.loc[valid_mask & (df_q[future_return_col] < ql), label_col] = -1
    df_q.loc[valid_mask & df_q[label_col].isna(), label_col] = 0

    print("\n=== L1_Q quantile-based parameters ===")
    print(f"k = {k}, q_low = {q_low}, q_high = {q_high}")
    print(f"Estimated train quantiles on V1-train R(t→t+k):")
    print(f"  q_low  (R <= q_low)  : {ql:.6f}")
    print(f"  q_high (R >= q_high) : {qh:.6f}")

    return df_q, ql, qh


# Run a single quantile-based configuration
Q_LOW = 0.2
Q_HIGH = 0.8

print("\n" + "=" * 80)
print(f"Running L1_Q quantile-based labels with k={L1_K_DEFAULT}, q_low={Q_LOW}, q_high={Q_HIGH}")
print("=" * 80)

df_l1_q, ql_L1Q, qh_L1Q = generate_L1_quantile_labels(
    df,
    train_start=V1_TRAIN_START,
    train_end=V1_TRAIN_END,
    k=L1_K_DEFAULT,
    q_low=Q_LOW,
    q_high=Q_HIGH,
    label_col="y_L1_Q",
    future_return_col="R_t_to_tk_Q",
)

# Label distribution by V1 segments + eras
l1_q_label_summary = summarize_label_distribution(
    df_l1_q,
    label_col="y_L1_Q",
    segments_dict=segments_for_labels,
)

# Run-length diagnostics on V1-train
l1_q_runs_v1_train = compute_run_lengths(
    df_l1_q,
    label_col="y_L1_Q",
    segment_name=f"V1_train (L1_Q q_low={Q_LOW}, q_high={Q_HIGH})",
    start=V1_TRAIN_START,
    end=V1_TRAIN_END,
)




Running L1_Q quantile-based labels with k=5, q_low=0.2, q_high=0.8

=== L1_Q quantile-based parameters ===
k = 5, q_low = 0.2, q_high = 0.8
Estimated train quantiles on V1-train R(t→t+k):
  q_low  (R <= q_low)  : -0.003375
  q_high (R >= q_high) : 0.003617

=== Label distribution for y_L1_Q by segment ===
       segment  label  count  proportion
      V1_train     -1    834    0.200048
      V1_train      0   2503    0.600384
      V1_train      1    832    0.199568
        V1_val     -1     87    0.083413
        V1_val      0    820    0.786194
        V1_val      1    136    0.130393
       V1_test     -1    208    0.135505
       V1_test      0   1123    0.731596
       V1_test      1    204    0.132899
Era1_2000_2004     -1    188    0.144615
Era1_2000_2004      0    944    0.726154
Era1_2000_2004      1    168    0.129231
Era2_2005_2009     -1    248    0.190184
Era2_2005_2009      0    790    0.605828
Era2_2005_2009      1    266    0.203988
Era3_2010_2014     -1    309    0.23

In [ ]:
# %%
# ===== Finalize baseline L1_Q labels (k=5, q_low=0.2, q_high=0.8) =====

# Recompute L1_Q on the full cleaned series, in a clean way
Q_LOW_BASELINE = 0.2
Q_HIGH_BASELINE = 0.8

print("\n" + "=" * 80)
print(f"Final baseline L1_Q: k={L1_K_DEFAULT}, q_low={Q_LOW_BASELINE}, q_high={Q_HIGH_BASELINE}")
print("=" * 80)

df_l1_q_final, ql_L1Q_final, qh_L1Q_final = generate_L1_quantile_labels(
    df,
    train_start=V1_TRAIN_START,
    train_end=V1_TRAIN_END,
    k=L1_K_DEFAULT,
    q_low=Q_LOW_BASELINE,
    q_high=Q_HIGH_BASELINE,
    label_col="y_L1_Q",
    future_return_col="R_t_to_tk_Q",
)

# Save baseline L1_Q labels to a dedicated file
l1_q_out_path = DATA_DERIVED / "mru_usd_labels_L1_Q_k5_q20_80.csv"
cols_l1_q_save = ["date", "close", "log_return", "R_t_to_tk_Q", "y_L1_Q"]
df_l1_q_final[cols_l1_q_save].to_csv(l1_q_out_path, index=False)

print(f"\nSaved baseline L1_Q labels to: {l1_q_out_path}")

# Compact label distribution summary (V1 segments only) for documentation
segments_v1_only = {
    "V1_train": (V1_TRAIN_START, V1_TRAIN_END),
    "V1_val":   (V1_VAL_START, V1_VAL_END),
    "V1_test":  (V1_TEST_START, V1_TEST_END),
}

print("\n=== Baseline L1_Q – label distribution on V1 segments ===")
l1_q_v1_summary = summarize_label_distribution(
    df_l1_q_final,
    label_col="y_L1_Q",
    segments_dict=segments_v1_only,
)

# Run-length stats on V1_train for baseline L1_Q
l1_q_runs_v1_train_final = compute_run_lengths(
    df_l1_q_final,
    label_col="y_L1_Q",
    segment_name="V1_train (L1_Q baseline)",
    start=V1_TRAIN_START,
    end=V1_TRAIN_END,
)



Final baseline L1_Q: k=5, q_low=0.2, q_high=0.8

=== L1_Q quantile-based parameters ===
k = 5, q_low = 0.2, q_high = 0.8
Estimated train quantiles on V1-train R(t→t+k):
  q_low  (R <= q_low)  : -0.003375
  q_high (R >= q_high) : 0.003617

Saved baseline L1_Q labels to: ..\data\derived\mru_usd_labels_L1_Q_k5_q20_80.csv

=== Baseline L1_Q – label distribution on V1 segments ===

=== Label distribution for y_L1_Q by segment ===
 segment  label  count  proportion
V1_train     -1    834    0.200048
V1_train      0   2503    0.600384
V1_train      1    832    0.199568
  V1_val     -1     87    0.083413
  V1_val      0    820    0.786194
  V1_val      1    136    0.130393
 V1_test     -1    208    0.135505
 V1_test      0   1123    0.731596
 V1_test      1    204    0.132899

=== Run-length stats for y_L1_Q in V1_train (L1_Q baseline) ===
       count      mean        std  min  25%  50%  75%   max
label                                                      
-1     279.0  2.989247   2.340577  

In [ ]:
# %%
# =====  Summaries for σ-based L1 strict variant (k=5, λ=0.75) =====

# Path of the existing strict L1 file from the earlier Week 5 run
l1_strict_path = DATA_DERIVED / "mru_usd_labels_L1_k5_l075.csv"

print("\nLoading strict L1 (σ-based, k=5, λ=0.75) from:")
print(l1_strict_path)

df_l1_strict = pd.read_csv(l1_strict_path)
df_l1_strict["date"] = pd.to_datetime(df_l1_strict["date"])

# Sanity: ensure expected columns exist
expected_cols = {"date", "close", "log_return", "R_t_to_tk", "y_L1"}
missing = expected_cols - set(df_l1_strict.columns)
if missing:
    print("\n[WARNING] Missing expected columns in strict L1 file:", missing)

print("\n=== Strict L1 (λ=0.75) – label distribution on V1 segments ===")
l1_strict_v1_summary = summarize_label_distribution(
    df_l1_strict,
    label_col="y_L1",
    segments_dict=segments_v1_only,
)

# Run-length stats on V1_train for strict L1
l1_strict_runs_v1_train = compute_run_lengths(
    df_l1_strict,
    label_col="y_L1",
    segment_name="V1_train (L1 strict λ=0.75)",
    start=V1_TRAIN_START,
    end=V1_TRAIN_END,
)




Loading strict L1 (σ-based, k=5, λ=0.75) from:
..\data\derived\mru_usd_labels_L1_k5_l075.csv

=== Strict L1 (λ=0.75) – label distribution on V1 segments ===

=== Label distribution for y_L1 by segment ===
 segment  label  count  proportion
V1_train     -1    179    0.042936
V1_train      0   3812    0.914368
V1_train      1    178    0.042696
  V1_val     -1      6    0.005753
  V1_val      0   1029    0.986577
  V1_val      1      8    0.007670
 V1_test     -1     14    0.009121
 V1_test      0   1503    0.979153
 V1_test      1     18    0.011726

=== Run-length stats for y_L1 in V1_train (L1 strict λ=0.75) ===
       count       mean        std  min  25%  50%   75%    max
label                                                         
-1      73.0   2.452055   1.598968  1.0  1.0  2.0   4.0    5.0
 0     127.0  30.015748  70.578672  1.0  2.0  5.0  21.0  495.0
 1      72.0   2.472222   1.694832  1.0  1.0  2.0   4.0    7.0

Finished strict L1 (λ=0.75) summary.


In [ ]:
# %%
# =====  Additional diagnostics for L1 σ-based labels (lambda = 0.5) =====
#  recomputes L1 with k=5, lambda=0.5 and reports:
# - Label distribution on V1_train / V1_val / V1_test
# - Run-length statistics for each of these segments

L1_K_L050 = 5
L1_LAMBDA_L050 = 0.5

print("\n" + "=" * 80)
print(f"Running detailed diagnostics for L1 σ-based labels with k={L1_K_L050}, lambda={L1_LAMBDA_L050}")
print("=" * 80)

# Recompute L1 with lambda=0.5 (independent of any previous grid cell)
df_l1_l050, sigma_train_L1_l050, theta_L1_l050 = generate_L1_labels(
    df,
    train_start=V1_TRAIN_START,
    train_end=V1_TRAIN_END,
    k=L1_K_L050,
    lambda_=L1_LAMBDA_L050,
    label_col="y_L1_l050",
    future_return_col="R_t_to_tk_l050",
)

print("\n=== L1 (lambda=0.5) parameters ===")
print(f"k = {L1_K_L050}, lambda = {L1_LAMBDA_L050}")
print(f"sigma_train (V1-train) = {sigma_train_L1_l050:.6f}")
print(f"theta = lambda * sigma_train * sqrt(k) = {theta_L1_l050:.6f}")

# V1 segments only, for clean comparison with other configurations
segments_v1_l050 = {
    "V1_train": (V1_TRAIN_START, V1_TRAIN_END),
    "V1_val":   (V1_VAL_START, V1_VAL_END),
    "V1_test":  (V1_TEST_START, V1_TEST_END),
}

print("\n=== L1 (lambda=0.5) – label distribution on V1 segments ===")
l1_l050_v1_summary = summarize_label_distribution(
    df_l1_l050,
    label_col="y_L1_l050",
    segments_dict=segments_v1_l050,
)

# Run-length stats for each V1 segment
l1_l050_runs = {}
for seg_name, (seg_start, seg_end) in segments_v1_l050.items():
    runs_df = compute_run_lengths(
        df_l1_l050,
        label_col="y_L1_l050",
        segment_name=f"{seg_name} (L1 lambda=0.5)",
        start=seg_start,
        end=seg_end,
    )
    l1_l050_runs[seg_name] = runs_df




Running detailed diagnostics for L1 σ-based labels with k=5, lambda=0.5

=== L1 parameters ===
k = 5, lambda = 0.5
sigma_train (V1-train) = 0.012360
theta = lambda * sigma_train * sqrt(k) = 0.013819

=== L1 (lambda=0.5) parameters ===
k = 5, lambda = 0.5
sigma_train (V1-train) = 0.012360
theta = lambda * sigma_train * sqrt(k) = 0.013819

=== L1 (lambda=0.5) – label distribution on V1 segments ===

=== Label distribution for y_L1_l050 by segment ===
 segment  label  count  proportion
V1_train     -1    300    0.071960
V1_train      0   3588    0.860638
V1_train      1    281    0.067402
  V1_val     -1     15    0.014382
  V1_val      0   1009    0.967402
  V1_val      1     19    0.018217
 V1_test     -1     23    0.014984
 V1_test      0   1474    0.960261
 V1_test      1     38    0.024756

=== Run-length stats for y_L1_l050 in V1_train (L1 lambda=0.5) ===
       count       mean        std  min  25%  50%    75%    max
label                                                          


In [ ]:
# %%
# ===== Helper for quantile-based L1_Q labels (k=5, q_low=0.2, q_high=0.8) =====

def generate_L1_quantile_labels(
    df: pd.DataFrame,
    train_start: pd.Timestamp,
    train_end: pd.Timestamp,
    k: int = 5,
    q_low: float = 0.2,
    q_high: float = 0.8,
    label_col: str = "y_L1_Q",
    future_return_col: str = "R_t_to_tk",
):
    """
    Generate quantile-based L1_Q labels:

    1. Ensure log_price, log_return in df.
    2. Compute future k-day log return:
         R_{t→t+k} = log_price.shift(-k) - log_price
    3. On V1-train only, compute quantiles:
         q_low  = Q_{q_low}(R)
         q_high = Q_{q_high}(R)
    4. Labels:
         +1 if R >= q_high
         -1 if R <= q_low
          0 otherwise
       (only where R is defined; last k rows get NaN in R and then no label)
    """
    df_q = ensure_log_price_and_return(df).copy()

    # Future k-day log return
    df_q[future_return_col] = df_q["log_price"].shift(-k) - df_q["log_price"]

    # Training window mask for quantiles
    train_mask = (df_q["date"] >= train_start) & (df_q["date"] <= train_end)
    valid_mask = df_q[future_return_col].notna()
    train_R = df_q.loc[train_mask & valid_mask, future_return_col]

    q_low_val = train_R.quantile(q_low)
    q_high_val = train_R.quantile(q_high)

    # Initialize labels
    df_q[label_col] = np.nan

    df_q.loc[valid_mask & (df_q[future_return_col] >= q_high_val), label_col] = 1
    df_q.loc[valid_mask & (df_q[future_return_col] <= q_low_val), label_col] = -1
    df_q.loc[valid_mask & df_q[label_col].isna(), label_col] = 0

    print("\n=== L1_Q quantile-based parameters ===")
    print(f"k = {k}, q_low = {q_low}, q_high = {q_high}")
    print("Estimated train quantiles on V1-train R(t→t+k):")
    print(f"  q_low  (R <= q_low)  : {q_low_val:.6f}")
    print(f"  q_high (R >= q_high) : {q_high_val:.6f}")

    return df_q, float(q_low_val), float(q_high_val)


In [ ]:
# %%
# ===== Generate + save L1_Q file and run diagnostics =====

print("\n" + "=" * 80)
print("Generating L1_Q quantile-based labels and saving to file")
print("=" * 80)

# Generate L1_Q labels using V1-train for quantiles, k=5, q_low=0.2, q_high=0.8
df_l1_Q, q_low_val, q_high_val = generate_L1_quantile_labels(
    df,
    train_start=V1_TRAIN_START,
    train_end=V1_TRAIN_END,
    k=L1_K_DEFAULT,
    q_low=0.2,
    q_high=0.8,
    label_col="y_L1_Q",
    future_return_col="R_t_to_tk",
)

# Save compact L1_Q label file for later modules (baseline L1)
l1_q_out_path = DATA_DERIVED / "mru_usd_labels_L1_Q_k5_q20_60_20.csv"
cols_l1_q_save = ["date", "close", "log_return", "R_t_to_tk", "y_L1_Q"]

df_l1_Q[cols_l1_q_save].to_csv(l1_q_out_path, index=False)
print(f"\nSaved L1_Q labels to: {l1_q_out_path}")

# --- Diagnostics: class balance and run-lengths on V1 segments + eras ---

segments_for_L1_Q = {
    "V1_train": (V1_TRAIN_START, V1_TRAIN_END),
    "V1_val":   (V1_VAL_START, V1_VAL_END),
    "V1_test":  (V1_TEST_START, V1_TEST_END),
}
segments_for_L1_Q.update(ERA_SEGMENTS)

l1_q_label_summary = summarize_label_distribution(
    df_l1_Q,
    label_col="y_L1_Q",
    segments_dict=segments_for_L1_Q,
)

l1_q_runs_v1_train = compute_run_lengths(
    df_l1_Q,
    label_col="y_L1_Q",
    segment_name="V1_train",
    start=V1_TRAIN_START,
    end=V1_TRAIN_END,
)


print(f"  {l1_q_out_path}")



Generating L1_Q quantile-based labels and saving to file

=== L1_Q quantile-based parameters ===
k = 5, q_low = 0.2, q_high = 0.8
Estimated train quantiles on V1-train R(t→t+k):
  q_low  (R <= q_low)  : -0.003375
  q_high (R >= q_high) : 0.003617

Saved L1_Q labels to: ..\data\derived\mru_usd_labels_L1_Q_k5_q20_60_20.csv

=== Label distribution for y_L1_Q by segment ===
       segment  label  count  proportion
      V1_train     -1    834    0.200048
      V1_train      0   2499    0.599424
      V1_train      1    836    0.200528
        V1_val     -1     87    0.083413
        V1_val      0    820    0.786194
        V1_val      1    136    0.130393
       V1_test     -1    208    0.135505
       V1_test      0   1123    0.731596
       V1_test      1    204    0.132899
Era1_2000_2004     -1    188    0.144615
Era1_2000_2004      0    944    0.726154
Era1_2000_2004      1    168    0.129231
Era2_2005_2009     -1    248    0.190184
Era2_2005_2009      0    790    0.605828
Era2_2005_2

In [ ]:
# Creates PNG figures for Week 5 and saves them under results/week05_figures/
#
# Inputs (as produced in Week 5):
# - data/processed/mru_usd_ohlc_clean.csv
# - data/derived/mru_usd_labels_L2_ma10_50_h5.csv
# - data/derived/mru_usd_labels_L1_Q_k5_q20_60_20.csv   (BASELINE L1_Q)
# - data/derived/mru_usd_features_all_week5.csv
#
# Figures saved (high-value, Week-5-native):
# 1) close_price_cleaned.png
# 2) log_return_hist.png
# 3) labels_overlay_L1Q_baseline.png
# 4) labels_overlay_L2.png
# 5) vol10_and_highvol_threshold.png
# 6) core_feature_corr_heatmap_v1train.png

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


# ===== Week 5 constants (as executed) =====
V1_TRAIN_START = pd.Timestamp("2000-01-03")
V1_TRAIN_END   = pd.Timestamp("2015-12-31")

CORE_FEATURE_COLS = [
    "f_r_lag1",
    "f_past_ret_5",
    "f_past_ret_20",
    "f_vol_10",
    "f_vol_30",
]

L1Q_BASELINE_FILE = "mru_usd_labels_L1_Q_k5_q20_60_20.csv"
L2_FILE = "mru_usd_labels_L2_ma10_50_h5.csv"
FEATURES_FILE = "mru_usd_features_all_week5.csv"
CLEAN_FILE = "mru_usd_ohlc_clean.csv"


# ===== Helpers =====
def find_project_root(start: Path) -> Path:

    cur = start.resolve()
    # for _ in range(12):
    #     if (cur / "data").exists():
    #         return cur
    #     if cur.parent == cur:
    #         break
    #     cur = cur.parent
    # raise FileNotFoundError("Could not locate project root containing a 'data/' directory.")


def ensure_log_price_and_return(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "log_price" not in df.columns:
        df["log_price"] = np.log(df["close"])
    if "log_return" not in df.columns:
        df["log_return"] = df["log_price"].diff()
    return df


def require_columns(df: pd.DataFrame, cols: list[str], name: str) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"{name} missing columns: {missing}")


def savefig(out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


def setup_time_axis(ax: plt.Axes) -> None:
    ax.xaxis.set_major_locator(mdates.YearLocator(base=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_minor_locator(mdates.YearLocator(base=1))
    ax.grid(True, which="major", linestyle="--", alpha=0.3)
    ax.grid(True, which="minor", linestyle=":", alpha=0.15)


def plot_close_series(df_clean: pd.DataFrame, out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(df_clean["date"], df_clean["close"], linewidth=1.2)
    ax.set_title("MRU/USD Close Price (Cleaned Series)")
    ax.set_xlabel("Date")
    ax.set_ylabel("Close (MRU/USD)")
    setup_time_axis(ax)
    savefig(out_path)


def plot_log_return_hist(df_clean: pd.DataFrame, out_path: Path) -> None:
    df_clean = ensure_log_price_and_return(df_clean)
    x = df_clean["log_return"].dropna().values
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(x, bins=100)
    ax.set_title("Histogram of Daily Log Returns (Cleaned Series)")
    ax.set_xlabel("Daily log return")
    ax.set_ylabel("Count")
    ax.grid(True, linestyle="--", alpha=0.3)
    savefig(out_path)


def plot_label_overlay(
    df_clean: pd.DataFrame,
    df_labels: pd.DataFrame,
    label_col: str,
    out_path: Path,
    title: str,
) -> None:
    require_columns(df_clean, ["date", "close"], "clean data")
    require_columns(df_labels, ["date", label_col], f"labels ({label_col})")

    base = df_clean[["date", "close"]].copy()
    lab = df_labels[["date", label_col]].copy()
    merged = base.merge(lab, on="date", how="left")

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(merged["date"], merged["close"], linewidth=1.0, alpha=0.6, label="Close")

    # Color-coded points for labels (NaNs ignored)
    colors = {-1: "#d62728", 0: "#7f7f7f", 1: "#2ca02c"}  # down / neutral / up
    names = {-1: "Down (-1)", 0: "Neutral (0)", 1: "Up (+1)"}

    for v in (-1, 0, 1):
        m = merged[label_col] == v
        if m.any():
            ax.scatter(
                merged.loc[m, "date"],
                merged.loc[m, "close"],
                s=8,
                alpha=0.7,
                c=colors[v],
                label=names[v],
                linewidths=0,
            )

    ax.set_title(title)
    ax.set_xlabel("Date")
    ax.set_ylabel("Close (MRU/USD)")
    setup_time_axis(ax)
    ax.legend(loc="upper left", ncol=3, frameon=True)
    savefig(out_path)


def plot_vol10_with_threshold(
    df_features: pd.DataFrame,
    out_path: Path,
    q: float = 0.7,
    vol_col: str = "f_vol_10",
) -> None:
    require_columns(df_features, ["date", vol_col], "features")
    df = df_features.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date")

    # Threshold computed exactly as Week 5: quantile on V1-train only
    mask_train = (df["date"] >= V1_TRAIN_START) & (df["date"] <= V1_TRAIN_END)
    thr = df.loc[mask_train, vol_col].dropna().quantile(q)

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(df["date"], df[vol_col], linewidth=1.0)
    ax.axhline(thr, linestyle="--", linewidth=1.2, label=f"V1-train {int(q*100)}th pct = {thr:.6f}")
    ax.set_title("10-day Rolling Volatility of Log Returns (f_vol_10) + High-Vol Threshold")
    ax.set_xlabel("Date")
    ax.set_ylabel("Rolling σ (log returns)")
    setup_time_axis(ax)
    ax.legend(loc="upper right", frameon=True)
    savefig(out_path)


def plot_core_corr_heatmap_v1train(
    df_features: pd.DataFrame,
    out_path: Path,
) -> None:
    require_columns(df_features, ["date"] + CORE_FEATURE_COLS, "features")
    df = df_features.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date")

    mask_train = (df["date"] >= V1_TRAIN_START) & (df["date"] <= V1_TRAIN_END)
    train = df.loc[mask_train, CORE_FEATURE_COLS].dropna()
    if train.empty:
        raise ValueError("No complete rows for core features in V1-train; cannot compute correlation heatmap.")

    corr = train.corr()

    fig, ax = plt.subplots(figsize=(7.5, 6))
    im = ax.imshow(corr.values, vmin=-1.0, vmax=1.0)
    ax.set_title("Core Feature Correlation Matrix (V1-train)")

    ax.set_xticks(range(len(CORE_FEATURE_COLS)))
    ax.set_yticks(range(len(CORE_FEATURE_COLS)))
    ax.set_xticklabels(CORE_FEATURE_COLS, rotation=45, ha="right")
    ax.set_yticklabels(CORE_FEATURE_COLS)

    # annotate
    for i in range(corr.shape[0]):
        for j in range(corr.shape[1]):
            ax.text(j, i, f"{corr.iloc[i, j]:.3f}", ha="center", va="center", fontsize=8)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Correlation")
    savefig(out_path)


def main() -> int:
    root = find_project_root(Path.cwd())

    data_processed = root / "data" / "processed"
    data_derived = root / "data" / "derived"
    results_dir = root / "results" / "week05_figures"
    results_dir.mkdir(parents=True, exist_ok=True)

    clean_path = data_processed / CLEAN_FILE
    l1q_path = data_derived / L1Q_BASELINE_FILE
    l2_path = data_derived / L2_FILE
    feat_path = data_derived / FEATURES_FILE

    for p in [clean_path, l1q_path, l2_path, feat_path]:
        if not p.exists():
            print(f"[ERROR] Missing required input file: {p}")
            return 2

    # Load inputs
    df_clean = pd.read_csv(clean_path)
    require_columns(df_clean, ["date", "close"], "clean data")
    df_clean["date"] = pd.to_datetime(df_clean["date"])
    df_clean = df_clean.sort_values("date").reset_index(drop=True)

    df_l1q = pd.read_csv(l1q_path)
    df_l1q["date"] = pd.to_datetime(df_l1q["date"])

    df_l2 = pd.read_csv(l2_path)
    df_l2["date"] = pd.to_datetime(df_l2["date"])

    df_feat = pd.read_csv(feat_path)
    df_feat["date"] = pd.to_datetime(df_feat["date"])

    # 1) Clean close series
    plot_close_series(df_clean, results_dir / "close_price_cleaned.png")

    # 2) Log-return histogram
    plot_log_return_hist(df_clean, results_dir / "log_return_hist.png")

    # 3) Baseline L1_Q overlay 
    plot_label_overlay(
        df_clean=df_clean,
        df_labels=df_l1q,
        label_col="y_L1_Q",
        out_path=results_dir / "labels_overlay_L1Q_baseline.png",
        title="Baseline L1_Q Regime Labels vs Close Price (k=5, q20/q80; saved file: q20_60_20)",
    )

    # 4) L2 overlay
    plot_label_overlay(
        df_clean=df_clean,
        df_labels=df_l2,
        label_col="y_L2",
        out_path=results_dir / "labels_overlay_L2.png",
        title="L2 Regime Labels vs Close Price (MA10/MA50 + slope h=5)",
    )

    # 5) f_vol_10 with high-vol threshold (computed on V1-train, q=0.7)
    plot_vol10_with_threshold(df_feat, results_dir / "vol10_and_highvol_threshold.png", q=0.7, vol_col="f_vol_10")

    # 6) Core correlation heatmap on V1-train
    plot_core_corr_heatmap_v1train(df_feat, results_dir / "core_feature_corr_heatmap_v1train.png")

    print(f"[OK] Saved Week 5 thesis figures to: {results_dir}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
